# Phase 1: Train the plant disease classifier

MobileNetV2 transfer learning on PlantVillage, evaluated on a held-out test set, exported to ONNX.

**Kaggle setup (do this before Run All)**
1. Settings > Accelerator: **GPU** (P100 or T4; one GPU is used).
2. Settings > Internet: **On** (needed for `pip install` and the pretrained MobileNetV2 weights).
3. Add Input > search **PlantVillage** and add a dataset that has one folder per class (38 classes). If it has `color`, `grayscale` and `segmented` folders, the notebook picks `color`.
4. Run All. Outputs land in `/kaggle/working/export`, `/kaggle/working/evaluation` and `phase1_outputs.zip`.

**Outputs that go into the repo**
- `ml/export/model.onnx`, `ml/export/labels.json`, `ml/export/preprocessing.json`
- `ml/evaluation/` (metrics, confusion matrix, per-class report, training curves)

In [ ]:
!pip install -q onnx onnxruntime onnxscript

## 1. Config and imports
Every tunable value lives in this one cell. `CONF_THRESHOLD` is copied from PROJECT_CONTEXT.md section 8 and is only used here to *check* the threshold, not to change it.

In [ ]:
import os, json, time, random, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# ---- settings you may want to change ----
DATA_DIR = None                  # None = auto-detect under /kaggle/input, or set the class-folder root by hand
MODEL_VERSION = "mnv2-plantvillage-v1"
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 64
HEAD_EPOCHS = 2                  # phase A: only the new classifier head trains
FINETUNE_EPOCHS = 6              # phase B: whole network trains at a low learning rate
HEAD_LR = 1e-3
FINETUNE_LR = 1e-4
LABEL_SMOOTHING = 0.1
CONF_THRESHOLD = 0.60            # PROJECT_CONTEXT.md section 8
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_EXTS = {".jpg", ".jpeg", ".png"}

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
EXPORT_DIR = OUT_DIR / "export"
EVAL_DIR = OUT_DIR / "evaluation"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
NUM_WORKERS = min(4, os.cpu_count() or 2)
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device:", device, "| workers:", NUM_WORKERS)
if device.type == "cpu":
    print("WARNING: no GPU found. Turn on a GPU accelerator in the Kaggle settings, training on CPU will take hours.")

## 2. Find the dataset and build the file table
The finder looks for the folder whose sub-folders are class folders full of images, and prefers a path containing `color`. The printout tells you what it picked; if it picked wrong, set `DATA_DIR` in the config cell.

In [ ]:
def find_class_root(search_root="/kaggle/input"):
    best, best_score = None, (0, 0)
    for dirpath, dirnames, _ in os.walk(search_root):
        n_class_dirs = 0
        for d in dirnames:
            try:
                if any(Path(f).suffix.lower() in IMG_EXTS for f in os.listdir(os.path.join(dirpath, d))):
                    n_class_dirs += 1
            except OSError:
                pass
        score = (n_class_dirs, int("color" in dirpath.lower()))
        if n_class_dirs >= 10 and score > best_score:
            best, best_score = dirpath, score
    return best


root = DATA_DIR or find_class_root()
if root is None:
    raise RuntimeError("No class-folder dataset found under /kaggle/input. Add a PlantVillage dataset or set DATA_DIR.")
root = Path(root)
print("Using dataset root:", root)

classes = sorted(d.name for d in root.iterdir() if d.is_dir())
class_to_idx = {c: i for i, c in enumerate(classes)}

rows = []
for c in classes:
    for p in (root / c).iterdir():
        if p.suffix.lower() in IMG_EXTS:
            rows.append((str(p), class_to_idx[c], c))
df = pd.DataFrame(rows, columns=["path", "label", "class_name"])

print(f"{len(classes)} classes, {len(df)} images")
if len(classes) != 38:
    print("NOTE: PlantVillage normally has 38 classes. Check that the right folder was picked.")
print(df["class_name"].value_counts().tail(5))

## 3. Train / val / test split (70 / 15 / 15)
PlantVillage contains several photos derived from the same original leaf photo (same id before `___` in the file name, plus flipped or rotated copies in some versions). A plain random split would put near-duplicates in both train and test and inflate the score. So images are grouped by that id, and whole **groups** are assigned to a split, stratified by class. This reduces leakage but cannot remove it entirely (the same physical leaf can still be photographed under different ids).

In [ ]:
def group_key(path):
    stem = Path(path).stem
    return stem.split("___")[0] if "___" in stem else stem


df["group"] = df["class_name"] + "/" + df["path"].map(group_key)
if df.groupby("group").size().max() > 50:
    print("Group keys look unreliable for this dataset, falling back to one group per image.")
    df["group"] = df["path"]
print(f"{len(df)} images in {df['group'].nunique()} groups")

groups = df.groupby("group").agg(label=("label", "first")).reset_index()
g_train, g_tmp = train_test_split(groups, test_size=0.30, stratify=groups["label"], random_state=SEED)
g_val, g_test = train_test_split(g_tmp, test_size=0.50, stratify=g_tmp["label"], random_state=SEED)

split_of = {}
for name, g in (("train", g_train), ("val", g_val), ("test", g_test)):
    split_of.update({k: name for k in g["group"]})
df["split"] = df["group"].map(split_of)

print(df["split"].value_counts())
missing = [s for s in ("train", "val", "test") if df[df["split"] == s]["label"].nunique() != len(classes)]
assert not missing, f"Some classes are missing from splits: {missing}"

## 4. Transforms and data loaders
- **Eval transform** (val, test, and later the backend): resize straight to 224x224, scale to 0-1, normalise with ImageNet mean/std. This exact recipe is written to `preprocessing.json`.
- **Train transform**: same normalisation, plus random crop, flips, rotation and colour jitter. The jitter and crops are a cheap attempt to help with the lab-photo vs phone-photo gap noted in PROJECT_CONTEXT.md section 9.

In [ ]:
normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03),
    transforms.ToTensor(),
    normalize,
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize,
])


class LeafDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]


def make_loader(split, transform, shuffle):
    ds = LeafDataset(df[df["split"] == split], transform)
    return DataLoader(ds, batch_size=BATCH_SIZE if shuffle else 128, shuffle=shuffle, drop_last=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)


train_loader = make_loader("train", train_tf, True)
val_loader = make_loader("val", eval_tf, False)
test_loader = make_loader("test", eval_tf, False)
print("batches per epoch:", len(train_loader), "| val:", len(val_loader), "| test:", len(test_loader))

## 5. Model
MobileNetV2 with ImageNet weights; the final layer is replaced with a new head for our classes.

In [ ]:
def build_model(num_classes):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    in_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_features, num_classes))
    return m


model = build_model(len(classes)).to(device)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 6. Training
Two phases:
- **A (head only, 2 epochs):** the pretrained backbone is frozen (and its batch-norm layers kept in eval mode) so the random new head does not wreck good features.
- **B (fine-tune, 6 epochs):** everything trains at a 10x lower learning rate with a cosine schedule.

The checkpoint with the best **validation** accuracy is kept. The test set is not touched until section 7.

In [ ]:
train_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
eval_criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best = {"val_acc": 0.0, "epoch": None, "state": None}


def run_epoch(loader, criterion, optimizer=None, head_only=False):
    training = optimizer is not None
    model.train(training)
    if training and head_only:
        model.features.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n


def fit(epochs, optimizer, scheduler=None, head_only=False, tag=""):
    for _ in range(epochs):
        t0 = time.time()
        tl, ta = run_epoch(train_loader, train_criterion, optimizer, head_only)
        vl, va = run_epoch(val_loader, eval_criterion)
        if scheduler is not None:
            scheduler.step()
        for k, v in zip(history, (tl, ta, vl, va)):
            history[k].append(v)
        ep = len(history["val_acc"])
        if va > best["val_acc"]:
            best.update(val_acc=va, epoch=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        print(f"[{tag}] epoch {ep:2d} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f} | {time.time() - t0:.0f}s")


# Phase A: head only
for p in model.features.parameters():
    p.requires_grad = False
opt = torch.optim.Adam(model.classifier.parameters(), lr=HEAD_LR)
fit(HEAD_EPOCHS, opt, head_only=True, tag="head")

# Phase B: fine-tune everything
for p in model.parameters():
    p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=FINETUNE_LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FINETUNE_EPOCHS)
fit(FINETUNE_EPOCHS, opt, scheduler=sched, tag="finetune")

torch.save(best["state"], OUT_DIR / "best_model.pt")
print(f"Best val acc {best['val_acc']:.4f} at epoch {best['epoch']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs_x = range(1, len(history["val_acc"]) + 1)
for ax, key, title in ((axes[0], "loss", "Loss"), (axes[1], "acc", "Accuracy")):
    ax.plot(epochs_x, history[f"train_{key}"], label="train")
    ax.plot(epochs_x, history[f"val_{key}"], label="val")
    ax.axvline(HEAD_EPOCHS + 0.5, color="gray", linestyle="--", linewidth=1)
    ax.set_title(title); ax.set_xlabel("epoch"); ax.legend()
fig.tight_layout()
fig.savefig(EVAL_DIR / "training_curves.png", dpi=150)
plt.show()

## 7. Evaluation on the held-out test set
Reported: accuracy, macro F1 (fairer than accuracy because classes are imbalanced), top-3 accuracy, per-class report, confusion matrix, most common confusions, and a check of the 0.60 confidence threshold.

**Read the numbers honestly:** the test set comes from the same lab-style distribution as training, so it is optimistic for real phone photos (PROJECT_CONTEXT.md section 9). Report it as "accuracy on PlantVillage held-out images", not as real-world accuracy.

In [ ]:
model.load_state_dict(best["state"])
model.to(device).eval()

all_probs, all_y = [], []
with torch.no_grad():
    for x, y in test_loader:
        logits = model(x.to(device, non_blocking=True))
        all_probs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
        all_y.append(y.numpy())
probs = np.concatenate(all_probs)
y_true = np.concatenate(all_y)
y_pred = probs.argmax(1)
label_ids = list(range(len(classes)))

acc = float((y_pred == y_true).mean())
macro_f1 = float(f1_score(y_true, y_pred, average="macro"))
top3 = np.argsort(-probs, axis=1)[:, :3]
top3_acc = float(np.mean([y in row for y, row in zip(y_true, top3)]))
print(f"Test images: {len(y_true)}")
print(f"Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f} | Top-3 accuracy: {top3_acc:.4f}")

report_txt = classification_report(y_true, y_pred, labels=label_ids, target_names=classes, digits=4, zero_division=0)
(EVAL_DIR / "classification_report.txt").write_text(report_txt)
report = classification_report(y_true, y_pred, labels=label_ids, target_names=classes, output_dict=True, zero_division=0)
worst = sorted((report[c]["f1-score"], c) for c in classes)[:5]
print("\nWeakest classes by F1:")
for f1, c in worst:
    print(f"  {f1:.4f}  {c}")

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=label_ids)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(label_ids); ax.set_yticks(label_ids)
ax.set_xticklabels(classes, rotation=90, fontsize=6)
ax.set_yticklabels(classes, fontsize=6)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix (row-normalised), test set")
fig.colorbar(im, fraction=0.03)
fig.tight_layout()
fig.savefig(EVAL_DIR / "confusion_matrix.png", dpi=150)
plt.show()

off = cm.copy()
np.fill_diagonal(off, 0)
pairs = sorted(((int(off[i, j]), classes[i], classes[j]) for i, j in zip(*np.nonzero(off))), reverse=True)
print("Most common confusions (count, true -> predicted):")
for n, t, p in pairs[:10]:
    print(f"  {n:3d}  {t}  ->  {p}")

In [ ]:
conf = probs.max(1)
wrong = y_pred != y_true
rows = []
for thr in (0.4, 0.5, CONF_THRESHOLD, 0.7, 0.8, 0.9):
    confident = conf >= thr
    rows.append({
        "threshold": thr,
        "share_of_images_confident": round(float(confident.mean()), 4),
        "accuracy_when_confident": round(float((y_pred[confident] == y_true[confident]).mean()), 4) if confident.any() else None,
        "share_of_errors_flagged_uncertain": round(float((~confident & wrong).sum() / wrong.sum()), 4) if wrong.any() else None,
    })
threshold_table = pd.DataFrame(rows)
print(threshold_table.to_string(index=False))
print(f"\nMean confidence: correct = {conf[~wrong].mean():.3f}, wrong = {conf[wrong].mean():.3f}" if wrong.any() else "\nNo errors on the test set.")

metrics = {
    "model_version": MODEL_VERSION,
    "dataset_root": str(root),
    "num_classes": len(classes),
    "split_images": df["split"].value_counts().to_dict(),
    "best_val_acc": best["val_acc"],
    "best_epoch": best["epoch"],
    "test_accuracy": acc,
    "test_macro_f1": macro_f1,
    "test_top3_accuracy": top3_acc,
    "confidence_threshold_checked": CONF_THRESHOLD,
    "threshold_table": rows,
    "config": {
        "img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "head_epochs": HEAD_EPOCHS, "finetune_epochs": FINETUNE_EPOCHS,
        "head_lr": HEAD_LR, "finetune_lr": FINETUNE_LR, "label_smoothing": LABEL_SMOOTHING, "seed": SEED,
    },
    "versions": {"torch": torch.__version__, "torchvision": torchvision.__version__},
    "caveat": "Test images come from the same lab-style PlantVillage distribution; expect lower accuracy on real phone photos.",
}
(EVAL_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

## 8. Export to ONNX
- The exported graph ends with a **softmax**, so the backend reads probabilities directly and never has to remember to apply softmax.
- Batch size is dynamic.
- `labels.json` is a plain list: position `i` is the label for output index `i`. Labels are the raw PlantVillage folder names.
- `preprocessing.json` is the recipe the backend must copy exactly.

In [ ]:
class ExportWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return torch.softmax(self.m(x), dim=1)


export_model = ExportWrapper(model).to("cpu").eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path = EXPORT_DIR / "model.onnx"
export_kwargs = dict(
    input_names=["input"], output_names=["probabilities"],
    dynamic_axes={"input": {0: "batch"}, "probabilities": {0: "batch"}},
    opset_version=17,
)
try:
    torch.onnx.export(export_model, dummy, str(onnx_path), dynamo=False, **export_kwargs)   # newer torch
except TypeError:
    torch.onnx.export(export_model, dummy, str(onnx_path), **export_kwargs)                  # older torch

(EXPORT_DIR / "labels.json").write_text(json.dumps(classes, indent=2))

preprocessing = {
    "model_version": MODEL_VERSION,
    "input_name": "input",
    "input_layout": "NCHW",
    "input_shape": ["batch", 3, IMG_SIZE, IMG_SIZE],
    "input_dtype": "float32",
    "color_mode": "RGB",
    "resize": {"width": IMG_SIZE, "height": IMG_SIZE, "method": "bilinear", "keep_aspect_ratio": False,
               "reference": "PIL.Image.resize((224, 224), Image.Resampling.BILINEAR)"},
    "divide_by": 255.0,
    "normalize": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    "steps": ["open image", "convert to RGB", "resize to 224x224 (bilinear)", "float32 / 255",
              "(x - mean) / std per channel", "HWC -> CHW", "add batch dimension"],
    "output_name": "probabilities",
    "output_note": "softmax already applied; shape (batch, num_classes); index order matches labels.json",
    "num_classes": len(classes),
}
(EXPORT_DIR / "preprocessing.json").write_text(json.dumps(preprocessing, indent=2))

print("model.onnx size:", round(onnx_path.stat().st_size / 1e6, 1), "MB")

## 9. Verify the export
Loads `model.onnx` in ONNX Runtime, preprocesses 300 test images with plain PIL + NumPy following `preprocessing.json` (this is what the backend will do), and compares against the PyTorch model. If this passes, the exported file and the preprocessing recipe agree.

In [ ]:
import onnx
import onnxruntime as ort

onnx.checker.check_model(onnx.load(str(onnx_path)))

MEAN = np.array(IMAGENET_MEAN, dtype=np.float32).reshape(1, 1, 3)
STD = np.array(IMAGENET_STD, dtype=np.float32).reshape(1, 1, 3)


def preprocess_numpy(path):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = (arr - MEAN) / STD
    return arr.transpose(2, 0, 1)


sample = df[df["split"] == "test"].sample(300, random_state=SEED)
batch = np.stack([preprocess_numpy(p) for p in sample["path"]]).astype(np.float32)

sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
ort_probs = np.concatenate([sess.run(None, {"input": batch[i:i + 50]})[0] for i in range(0, len(batch), 50)])
with torch.no_grad():
    torch_probs = np.concatenate([export_model(torch.from_numpy(batch[i:i + 50])).numpy() for i in range(0, len(batch), 50)])

agree = float((ort_probs.argmax(1) == torch_probs.argmax(1)).mean())
max_diff = float(np.abs(ort_probs - torch_probs).max())
ort_acc = float((ort_probs.argmax(1) == sample["label"].to_numpy()).mean())
print("ONNX output shape:", ort_probs.shape, "| rows sum to 1:", bool(np.allclose(ort_probs.sum(1), 1, atol=1e-4)))
print(f"ONNX vs PyTorch: prediction agreement {agree:.4f}, max probability difference {max_diff:.2e}")
print(f"ONNX accuracy on these 300 test images (numpy preprocessing): {ort_acc:.4f}")
assert agree >= 0.99 and max_diff < 1e-3, "ONNX output does not match PyTorch, do not use this export."
print("Export verified.")

## 10. Package the outputs
Creates `phase1_outputs.zip` with `export/` and `evaluation/`. Download it from the Kaggle Output tab (or the file browser) and unzip it into the repo's `ml/` folder. `best_model.pt` stays in `/kaggle/working` so you can re-export without retraining, but it is not part of the repo.

In [ ]:
zip_path = OUT_DIR / "phase1_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (EXPORT_DIR, EVAL_DIR):
        for f in sorted(folder.rglob("*")):
            if f.is_file():
                z.write(f, f.relative_to(OUT_DIR))
                print("added", f.relative_to(OUT_DIR))
print("\nDone:", zip_path)